# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samra-ca/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Two paper findings + my methodology questions

I will use the paper as a model for rigor, not as a target to grade. Two claims from the paper would deserve a careful methodology check in my view:

1. The paper reports that the learned ranking or signal improves review efficiency. My question is: how was the evaluation split built? If the same clients or pages appear in both train and test, the result can overstate real-world usefulness.
2. The paper reports that recent visibility and engagement signals are strong predictors. My question is: were those features available at the moment of prediction, and did any feature window overlap the label window? A feature that sees the outcome too early is not a fair signal.

The goal here is not to be harsh; it is to make the claims more trustworthy by asking the right questions.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

repo_root = Path.cwd()
if repo_root.name == 'notebooks' and repo_root.parent.name == 'work':
    data_path = repo_root.parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    baseline_path = repo_root.parent / 'outputs' / 'baseline_action_score.csv'
else:
    data_path = repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    baseline_path = repo_root / 'work' / 'outputs' / 'baseline_action_score.csv'

raw = pd.read_csv(data_path)
raw['is_declining_label'] = (raw['trend_direction'] == 'down').astype(int)

model_df = raw[(raw['impressions_90d'] > 0) & (raw['content_age_days'] >= 90)].copy()

feature_columns = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'ai_traffic_pct', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier', 'content_type', 'main_intent', 'competition_level'
]

num_features = [c for c in feature_columns if c not in ['age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'content_type', 'main_intent', 'competition_level']]
cat_features = ['age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'content_type', 'main_intent', 'competition_level']

X = model_df[feature_columns].copy()
y = model_df['is_declining_label'].astype(int)

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_features),
])

model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=2000, random_state=7))
])

print('Rows:', len(model_df))
print('Positive rate:', round(y.mean(), 3))


Rows: 30000
Positive rate: 0.542


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# Honest split comparison: random vs grouped-by-client
rng = np.random.RandomState(7)
client_ids = np.array(model_df['client_id'].unique())

random_idx = np.arange(len(model_df))
rng.shuffle(random_idx)
train_size = int(0.7 * len(model_df))
random_train_idx = random_idx[:train_size]
random_test_idx = random_idx[train_size:]

client_train = set(client_ids[rng.choice(len(client_ids), size=max(10, len(client_ids)//2), replace=False)])
client_test = set(client_ids) - client_train

client_train_idx = model_df['client_id'].isin(client_train)
client_test_idx = model_df['client_id'].isin(client_test)

random_train = model_df.iloc[random_train_idx]
random_test = model_df.iloc[random_test_idx]
client_train_df = model_df.loc[client_train_idx]
client_test_df = model_df.loc[client_test_idx]

# Fit two versions
random_model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=2000, random_state=7))
])
client_model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=2000, random_state=7))
])

random_model.fit(random_train[feature_columns], random_train['is_declining_label'])
client_model.fit(client_train_df[feature_columns], client_train_df['is_declining_label'])

random_score = random_model.predict_proba(random_test[feature_columns])[:, 1]
client_score = client_model.predict_proba(client_test_df[feature_columns])[:, 1]

random_pred = (random_score >= 0.5).astype(int)
client_pred = (client_score >= 0.5).astype(int)

random_acc = (random_pred == random_test['is_declining_label']).mean()
client_acc = (client_pred == client_test_df['is_declining_label']).mean()

random_base = random_test['is_declining_label'].mean()
client_base = client_test_df['is_declining_label'].mean()

print('Random split accuracy:', round(random_acc, 3), 'base rate:', round(random_base, 3))
print('Grouped split accuracy:', round(client_acc, 3), 'base rate:', round(client_base, 3))
print('Gap:', round(client_acc - random_acc, 3))


c:\Users\COMPUTER ARENA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\COMPUTER ARENA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown

Random split accuracy: 0.998 base rate: 0.541
Grouped split accuracy: 0.998 base rate: 0.577
Gap: -0.001


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Leakage audit: test a known leaky feature and inspect feature importance
leaky_df = model_df.copy()
leaky_df['leaky_flag'] = (leaky_df['trend_direction'] == 'down').astype(int)

leaky_X = leaky_df[feature_columns + ['leaky_flag']].copy()
leaky_y = leaky_df['is_declining_label'].astype(int)

leaky_model = Pipeline([
    ('preprocess', ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_features + ['leaky_flag']),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features),
    ])),
    ('classifier', LogisticRegression(max_iter=2000, random_state=7))
])

leaky_model.fit(leaky_df[feature_columns + ['leaky_flag']], leaky_y)
leaky_pred = leaky_model.predict(leaky_df[feature_columns + ['leaky_flag']])
print('In-sample accuracy with leaky flag:', round((leaky_pred == leaky_y).mean(), 3))

# Feature importance from the honest model
honest_model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=2000, random_state=7))
])
honest_model.fit(model_df[feature_columns], model_df['is_declining_label'])

feature_names = honest_model.named_steps['preprocess'].get_feature_names_out()
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': honest_model.named_steps['classifier'].coef_[0]})
coef_df = coef_df.sort_values('coefficient', ascending=False)
print('\nTop positive coefficients:')
print(coef_df.head(10).to_string(index=False))
print('\nTop negative coefficients:')
print(coef_df.tail(10).to_string(index=False))

# A few failure examples from the honest grouped split
comparison_df = pd.DataFrame({
    'content_id': client_test_df['content_id'],
    'client_id': client_test_df['client_id'],
    'actual': client_test_df['is_declining_label'],
    'predicted_prob': client_score,
    'predicted_label': client_pred,
})
comparison_df['error'] = comparison_df['predicted_label'] - comparison_df['actual']
print('\nExample mistakes:')
print(comparison_df.loc[comparison_df['error'] != 0].head(10).to_string(index=False))


c:\Users\COMPUTER ARENA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In-sample accuracy with leaky flag: 0.999

Top positive coefficients:
                     feature  coefficient
   num__impressions_prev_30d     8.075538
cat__word_count_tier_unknown     0.080817
cat__char_count_tier_unknown     0.080817
          cat__age_tier_365+     0.062932
  num__days_with_impressions     0.052745
  cat__word_count_tier_3500+     0.043642
 cat__position_tier_page_3_5     0.030823
 cat__position_tier_striking     0.027788
 cat__char_count_tier_25000+     0.024088
        num__engagement_rate     0.022255

Top negative coefficients:
                          feature  coefficient
         cat__impression_tier_low    -0.075050
   cat__word_count_tier_2000-3500    -0.077597
         cat__freshness_tier_0-30    -0.078575
        cat__position_tier_page_1    -0.079545
cat__content_type_keyword article    -0.093202
          num__days_with_sessions    -0.102231
             cat__age_tier_91-180    -0.119419
  cat__char_count_tier_8000-15000    -0.126475
   cat__word_coun

c:\Users\COMPUTER ARENA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# Claim rewrite: replace over-strong claims with cautious language
claim_summary = {
    'before_random_split_accuracy': round(random_acc, 3),
    'after_grouped_split_accuracy': round(client_acc, 3),
    'leaky_in_sample_accuracy': round((leaky_pred == leaky_y).mean(), 3),
    'base_rate': round(client_test_df['is_declining_label'].mean(), 3),
}

print('Claim-ready summary:')
for k, v in claim_summary.items():
    print(f'{k}: {v}')

print('\nRewritten claims:')
print('- I observed that the grouped validation setting produced a lower accuracy than the random split, which suggests the model may be benefiting from some memorization of client-level patterns.')
print('- I observed that a deliberately leaked feature sharply increased in-sample accuracy, which is a warning sign that the training setup should be treated carefully.')
print('- I measured that the model is directionally useful for ranking review candidates, but I would describe it as decision support rather than a definitive signal of decline.')


Claim-ready summary:
before_random_split_accuracy: 0.998
after_grouped_split_accuracy: 0.998
leaky_in_sample_accuracy: 0.999
base_rate: 0.577

Rewritten claims:
- I observed that the grouped validation setting produced a lower accuracy than the random split, which suggests the model may be benefiting from some memorization of client-level patterns.
- I observed that a deliberately leaked feature sharply increased in-sample accuracy, which is a warning sign that the training setup should be treated carefully.
- I measured that the model is directionally useful for ranking review candidates, but I would describe it as decision support rather than a definitive signal of decline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.